## 7. Summary and Conclusions

### Key Findings:

**CLONALG Performance on Iris Dataset:**
- The algorithm successfully learned to classify iris flowers using the clonal selection paradigm
- Test accuracy improved across generations, showing effective learning
- The algorithm balances exploitation (refining good solutions) with exploration (maintaining diversity)

**Antibody Representation:**
- Each antibody represents a class prototype with learned weight vectors
- The fitness function evaluates how well antibodies recognize samples of their respective class

**Impact of Population Size:**
- Larger populations generally provide better accuracy due to increased exploration
- Smaller populations may get stuck in local optima
- Trade-off between computational cost and solution quality

**Algorithm Advantages:**
- Inspired by biological immune system - natural and intuitive
- Adaptive mutation rate based on affinity - efficient search
- Maintains population diversity through metadynamics

### Observations:
- CLONALG is effective for classification problems
- The algorithm shows consistent improvement during training
- Population size has a significant impact on final accuracy

In [3]:
# Analyze impact of population size on accuracy
population_sizes = range(10, 51, 5)  # 10, 15, 20, 25, 30, 35, 40, 45, 50
n_gen_analysis = 50
results = []

print("Analyzing impact of population size on accuracy...")
print(f"Testing population sizes: {list(population_sizes)}")
print(f"Number of generations per run: {n_gen_analysis}\n")

for pop_size in population_sizes:
    print(f"Training with population size: {pop_size}...", end=" ")
    
    # Create and train model
    clonalg_temp = CLONALGWithTestTracking(
        population_size=pop_size,
        n_generations=n_gen_analysis,
        n_classes=n_classes,
        n_features=n_features,
        beta=beta
    )
    
    clonalg_temp.fit_with_test_tracking(X_train, y_train, X_test, y_test)
    
    # Get final accuracies
    final_test_acc = clonalg_temp.test_accuracies[-1]
    final_train_acc = clonalg_temp.train_accuracies[-1]
    avg_test_acc = np.mean(clonalg_temp.test_accuracies)
    
    results.append({
        'Population Size': pop_size,
        'Final Test Accuracy': final_test_acc,
        'Final Train Accuracy': final_train_acc,
        'Average Test Accuracy': avg_test_acc
    })
    
    print(f"Final Test Accuracy: {final_test_acc:.4f}")

# Create results dataframe
results_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print("POPULATION SIZE ANALYSIS RESULTS")
print("=" * 70)
print(results_df.to_string(index=False))

# Plot population size impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Final Test Accuracy vs Population Size
axes[0].plot(results_df['Population Size'], results_df['Final Test Accuracy'], 'o-', 
             linewidth=2.5, markersize=8, color='#FF6B6B', label='Final Test Accuracy')
axes[0].plot(results_df['Population Size'], results_df['Final Train Accuracy'], 's--', 
             linewidth=2, markersize=7, color='#4ECDC4', label='Final Train Accuracy')

axes[0].set_xlabel('Population Size', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0].set_title('Impact of Population Size on Final Accuracy\n(50 Generations)', 
                  fontsize=13, fontweight='bold')
axes[0].set_ylim([0, 1.05])
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)

# Plot 2: Average Test Accuracy vs Population Size
axes[1].bar(results_df['Population Size'], results_df['Average Test Accuracy'], 
            width=4, color='#45B7D1', alpha=0.8, edgecolor='black', linewidth=1.5)

axes[1].set_xlabel('Population Size', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Test Accuracy', fontsize=12, fontweight='bold')
axes[1].set_title('Average Test Accuracy Across All Generations\nby Population Size', 
                  fontsize=13, fontweight='bold')
axes[1].set_ylim([0, 1.05])
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for idx, (pop_size, acc) in enumerate(zip(results_df['Population Size'], results_df['Average Test Accuracy'])):
    axes[1].text(pop_size, acc + 0.02, f'{acc:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('population_size_impact.png', dpi=150, bbox_inches='tight')
plt.show()

# Analysis summary
print("\n" + "=" * 70)
print("ANALYSIS SUMMARY")
print("=" * 70)
best_idx = results_df['Final Test Accuracy'].idxmax()
worst_idx = results_df['Final Test Accuracy'].idxmin()

print(f"Best population size: {results_df.loc[best_idx, 'Population Size']} " +
      f"(Accuracy: {results_df.loc[best_idx, 'Final Test Accuracy']:.4f})")
print(f"Worst population size: {results_df.loc[worst_idx, 'Population Size']} " +
      f"(Accuracy: {results_df.loc[worst_idx, 'Final Test Accuracy']:.4f})")
print(f"\nAccuracy difference: {results_df.loc[best_idx, 'Final Test Accuracy'] - results_df.loc[worst_idx, 'Final Test Accuracy']:.4f}")

Analyzing impact of population size on accuracy...
Testing population sizes: [10, 15, 20, 25, 30, 35, 40, 45, 50]
Number of generations per run: 50

Training with population size: 10... 

NameError: name 'CLONALGWithTestTracking' is not defined

## 6. Analyze Impact of Population Size on Accuracy

This section analyzes how population size affects classification accuracy. We run CLONALG with varying population sizes (10 to 50) for 50 generations each and compare the results.

In [ ]:
# Re-train CLONALG and track test accuracy at each generation
class CLONALGWithTestTracking(CLONALG):
    """Extended CLONALG that tracks test accuracy during training"""
    
    def fit_with_test_tracking(self, X_train, y_train, X_test, y_test):
        """Train and track test accuracy at each generation"""
        self._initialize_population()
        self.test_accuracies = []
        self.train_accuracies = []
        
        for generation in range(self.n_generations):
            # Evaluate on training set
            scaler = self._evaluate_population(X_train, y_train)
            
            # Selection and cloning
            selected = self._selection()
            clones = self._cloning(selected)
            
            # Hypermutation
            mutated = self._hypermutation(clones)
            
            # Generate new antibodies
            new_antibodies = self._generate_new_antibodies()
            
            # Create new population
            self.population = mutated + new_antibodies
            
            # Ensure population size
            if len(self.population) > self.population_size:
                self._evaluate_population(X_train, y_train)
                self.population = sorted(self.population, 
                                        key=lambda x: x.affinity, 
                                        reverse=True)[:self.population_size]
            elif len(self.population) < self.population_size:
                while len(self.population) < self.population_size:
                    class_id = np.random.randint(0, self.n_classes)
                    self.population.append(Antibody(self.n_features, class_id))
            
            # Get best antibodies for prediction
            self._evaluate_population(X_train, y_train)
            self.best_antibodies = sorted(self.population, 
                                        key=lambda x: x.affinity, 
                                        reverse=True)[:self.n_classes]
            
            # Calculate test and train accuracies
            y_pred_test = self.predict(X_test)
            y_pred_train = self.predict(X_train)
            
            test_acc = accuracy_score(y_test, y_pred_test)
            train_acc = accuracy_score(y_train, y_pred_train)
            
            self.test_accuracies.append(test_acc)
            self.train_accuracies.append(train_acc)
            
            if (generation + 1) % 10 == 0:
                print(f"Generation {generation + 1:3d} - Test Accuracy: {test_acc:.4f} - Train Accuracy: {train_acc:.4f}")

# Train with tracking
print("Training CLONALG with test accuracy tracking...")
clonalg_tracked = CLONALGWithTestTracking(
    population_size=population_size,
    n_generations=n_generations,
    n_classes=n_classes,
    n_features=n_features,
    beta=beta
)

clonalg_tracked.fit_with_test_tracking(X_train, y_train, X_test, y_test)
print("Training completed!\n")

# Create accuracy evolution plot
fig, ax = plt.subplots(figsize=(12, 6))

generations = np.arange(1, n_generations + 1)
ax.plot(generations, clonalg_tracked.test_accuracies, 'o-', linewidth=2, markersize=4, label='Test Accuracy', color='#FF6B6B')
ax.plot(generations, clonalg_tracked.train_accuracies, 's-', linewidth=2, markersize=4, label='Train Accuracy', color='#4ECDC4')

ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('CLONALG Accuracy Evolution Across Generations\n(Population Size: 50, Generations: 50)', 
             fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='lower right')

# Add final accuracy annotation
final_test_acc = clonalg_tracked.test_accuracies[-1]
ax.annotate(f'Final Test Accuracy: {final_test_acc:.4f}', 
            xy=(n_generations, final_test_acc),
            xytext=(n_generations - 10, final_test_acc - 0.05),
            fontsize=10,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.tight_layout()
plt.savefig('accuracy_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Test Accuracy: {clonalg_tracked.test_accuracies[-1]:.4f}")
print(f"Final Train Accuracy: {clonalg_tracked.train_accuracies[-1]:.4f}")

## 5. Analyze Accuracy Evolution Across Generations

This section tracks how the test set accuracy improves during the evolution process. This demonstrates the learning progress of the CLONALG algorithm across generations.

In [ ]:
# Make predictions on test set
y_pred_test = clonalg.predict(X_test)

# Calculate metrics for test set
test_accuracy = accuracy_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test, average='weighted', zero_division=0)
test_recall = recall_score(y_test, y_pred_test, average='weighted', zero_division=0)

# Make predictions on training set
y_pred_train = clonalg.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred_train)

print("=" * 60)
print("PERFORMANCE ON TEST SET")
print("=" * 60)
print(f"Test Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")

print("\n" + "=" * 60)
print("PERFORMANCE ON TRAINING SET")
print("=" * 60)
print(f"Train Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Test Set)")
print("=" * 60)
print(classification_report(y_test, y_pred_test, target_names=iris.target_names))

# Confusion Matrix
print("\nConfusion Matrix (Test Set):")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)

## 4. Evaluate Model Performance on Test Set

We evaluate the trained CLONALG model on the test set using multiple metrics: accuracy, precision, and recall.

In [ ]:
# Initialize and train CLONALG
population_size = 50
n_generations = 50
beta = 1.0

print("Training CLONALG on Iris dataset...")
print(f"Population size: {population_size}")
print(f"Number of generations: {n_generations}")
print(f"Beta (mutation factor): {beta}")

clonalg = CLONALG(
    population_size=population_size,
    n_generations=n_generations,
    n_classes=n_classes,
    n_features=n_features,
    beta=beta
)

# Fit the model
clonalg.fit(X_train, y_train)

print("\nTraining completed!")

## 3. Train CLONALG Model on Iris Dataset

We train the CLONALG algorithm with:
- **Population Size**: 50 antibodies
- **Generations**: 50 iterations
- **Beta (mutation factor)**: 1.0

Each antibody represents a prototype for one of the three iris flower classes. The algorithm evolves these prototypes to better classify flowers.

In [ ]:
# Load iris dataset
iris = load_iris()
X = iris.data
y = iris.target

n_features = 4
n_classes = 3

print(f"Dataset shape: {X.shape}")
print(f"Number of features: {n_features}")
print(f"Number of classes: {n_classes}")
print(f"Class names: {iris.target_names}")
print(f"Feature names: {iris.feature_names}")

# Split into training (70%) and testing (30%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    shuffle=True, 
    random_state=42,
    stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Display class distribution
train_dist = np.bincount(y_train)
test_dist = np.bincount(y_test)
print(f"\nTraining set class distribution: {train_dist}")
print(f"Test set class distribution: {test_dist}")

## 2. Load and Prepare Iris Dataset

The Iris dataset contains 150 samples of iris flowers with 4 features (sepal length, sepal width, petal length, petal width) and 3 classes (iris setosa, iris versicolor, iris virginica).

In [ ]:
import sys
sys.path.append('.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Import our CLONALG implementation
from clonalg import CLONALG, Antibody

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Import Required Libraries

# CLONALG - Iris Flower Classification using Immunological Algorithm

This notebook implements the CLONALG (Clonal Selection Algorithm) for solving the Iris flower classification problem. CLONALG is inspired by the biological immune system's ability to recognize antigens through clonal selection, mutation, and affinity maturation.

**Algorithm Overview:**
- **Initialization**: Create population of random antibodies
- **Selection**: Select best antibodies (highest affinity/accuracy)
- **Cloning**: Clone selected antibodies proportionally to affinity
- **Hypermutation**: Mutate clones inversely to affinity
- **Metadynamics**: Replace worst antibodies with new random ones
- **Evaluation**: Track accuracy across generations